# Paper: Towards JITAI -



In [ ]:
import os
import sys
import regex as re
# If your current working directory is the notebooks directory, use this:
notebook_dir = os.getcwd()  # current working directory
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
model_path = os.path.abspath(os.path.join(notebook_dir, '..', 'model_pipeline'))

sys.path.append(parent_dir)
sys.path.append(src_path)
sys.path.append(model_path)

import glob
import pickle
from IPython.display import Markdown
from server_config import datapath, preprocessed_path, preprocessed_path_freezed, redcap_path

import pandas as pd
import numpy as np
import datetime as dt
from scipy.stats import entropy

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
# Enable experimental features in scikit-learn
from sklearn.experimental import enable_iterative_imputer  # ✅ Must be imported first
from sklearn.impute import IterativeImputer  # ✅ Now you can import it
import sklearn

    
import ML_config
import ML_pipeline
import run_ML_pipeline

import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns 
import matplotlib.patches as mpatches

sns.set_context("notebook", rc={"axes.labelsize": 14, "xtick.labelsize": 14, "ytick.labelsize": 14})
sns.set_style("whitegrid", {'axes.grid': True})
import plotly.express as px
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
import seaborn as sns
from scipy.stats import gaussian_kde

%matplotlib inline


In [ ]:
#backup_path = preprocessed_path + "backup_data_passive.feather"
#df_backup = pd.read_feather(backup_path)

with open(preprocessed_path_freezed + '/ema_data.pkl', 'rb') as file:
    df_ema_framework = pickle.load(file)

with open(preprocessed_path_freezed + '/ema_content.pkl', 'rb') as file:
    df_ema_content = pickle.load(file)  

with open(preprocessed_path_freezed + '/monitoring_data.pkl', 'rb') as file:
    df_monitoring = pickle.load(file)

with open(preprocessed_path_freezed + '/redcap_data.pkl', 'rb') as file:
    df_redcap = pickle.load(file)

with open(preprocessed_path_freezed + '/code_check'+'/map_ema_passive.pkl', 'rb') as file:
    df_ema_passive = pickle.load(file)

### Configurations

In [ ]:
# EMA
assessment_phase = [0] #1,2
min_num_daily = 4
min_days_data = 7

### Compare Included vs. not Included Participantants

In [ ]:
df_ema_content_cust = df_ema_content.customer.unique().tolist()

In [ ]:
df_redcap_original = df_redcap.dropna(subset = ["age", "customer"] )
df_redcap_original = df_redcap_original[df_redcap_original.customer.isin(df_ema_content_cust)]
df_redcap_original = df_redcap_original.drop_duplicates(subset="customer")


In [ ]:
# Create a set of included customer IDs
included_customers = set(df_ema_passive['customer'])

# Add a new column to df_redcap_original indicating inclusion
df_redcap_original['Included'] = df_redcap_original['customer'].isin(included_customers)

# Define the two groups
df_redcap_original['Group'] = df_redcap_original['Included'].map({True: 'Included', False: 'Not Included'})

# Verify the counts
print(f"Subjects included in the analysis (n={df_redcap_original['Group'].value_counts().get('Included', 0)})")
print(f"Subjects not included in the analysis (n={df_redcap_original['Group'].value_counts().get('Not Included', 0)})")


In [ ]:
from tableone import TableOne
# Define your variables
# Replace the variable names with those present in your DataFrame

# Demographic variables
age = 'age'  # Continuous
employable = 'employability_description_simple'  # Categorical
smartphone_type = 'ema_smartphone_description'  # Categorical
psychotropic_med = 'psychotropic_description'
diagnosis = 'scid_cv_description'
previous_treatment = 'prior_treatment_description_simple'
somatic = 'somatic_description'



# List of all variables to include in the table
columns = [age, employable, smartphone_type, previous_treatment, psychotropic_med, diagnosis, somatic]

# Define categorical variables
categorical = [employable, smartphone_type, previous_treatment, psychotropic_med, diagnosis, somatic]

# Define grouping variable
group_var = 'Included'


In [ ]:
# Create the TableOne object
table1 = TableOne(
    df_redcap_original,
    columns=columns,
    categorical=categorical,
    groupby=group_var,
    pval=True,
    nonnormal=[],  # Add variables that are non-normally distributed if any
    missing=False  # Whether to include missing data
)

# Print the table
print(table1.tabulate(tablefmt="fancy_grid"))
table1.to_csv('sample_overview.csv')


## Manual Missing data handling

In [ ]:
# also impute activity features 

#### GPS

In [ ]:
# Create a mask for rows where missing_GPS equals 'Steps<=625'

mask = df_ema_passive['missing_GPS'] == 'Steps<=625'

# For these rows, set the selected columns to 0
cols_set_zero = ['n_GPS', 'total_distance_km', 'time_in_transition_minutes']
for col in cols_set_zero:
    df_ema_passive.loc[mask, col] = 0

# For these rows, set the selected columns to 120
cols_set_120 = ['time_stationary_minutes']
for col in cols_set_120:
    df_ema_passive.loc[mask, col] = 120

mask = df_ema_passive['missing_GPS_home'] == 'Steps<=625'

# For these rows, set the selected columns to 120
cols_set_120 = ['at_home_minute']
for col in cols_set_120:
    df_ema_passive.loc[mask, col] = 120


#### Steps

In [ ]:

mask = df_ema_passive['missing_steps'] == 'step_zero'

# For these rows, set the selected columns to 0
cols_set_zero = ['n_steps']
for col in cols_set_zero:
    df_ema_passive.loc[mask, col] = 0



#### Physical Activity

In [ ]:
# Create a mask for rows where missing_GPS equals 'Steps>625'

mask = df_ema_passive['missing_pa'] == 'pa_zero'

# For these rows, set the selected columns to 0
cols_set_zero = ['activity_102_minutes', 'activity_103_minutes', 'activity_104_minutes', 'activity_105_minutes', 'activity_106_minutes', 
                 'activity_107_minutes']
for col in cols_set_zero:
    df_ema_passive.loc[mask, col] = 0



### Feature Encoding

- prior treatment: ordinal encoding
- age: min-max scaling
- somatic, employability, psychotropic: 

In [ ]:
# Define which columns are which
binary_features = ['somatic_description', 'psychotropic_description', 'employability_description_simple', 'smartphone_type', 'weekend']
categorical_features = ['weekday', 'prior_treatment_description_simple', 'quest_create_hour', 'season', 'time_of_day', 'employability_description_simple']
numeric_features = ['age','hr_mean', 'hr_min', 'hr_max', 'hr_std', 'hr_zone_resting', 'hr_zone_moderate','hr_zone_vigorous', 'n_steps', 
       'n_GPS', 'total_distance_km', 'at_home_minute',
       'time_in_transition_minutes', 'time_stationary_minutes', 'activity_102_minutes',
       'activity_103_minutes', 'activity_104_minutes', 'activity_105_minutes',
       'activity_106_minutes', 'activity_107_minutes',
       'apparent_temperature_mean', 'sunshine_duration', 'precipitation_hours'] 

person_static_features = ['customer', 'age', 'somatic_description', 'psychotropic_description', 'employability_description_simple', 'smartphone_type', 'weekend']


In [ ]:
categories = [sorted(df_ema_passive[feat].dropna().unique()) for feat in categorical_features]


In [ ]:
df_ema_passive[numeric_features] = df_ema_passive[numeric_features].replace(-1, np.nan)

In [ ]:
from scipy.stats import skewtest,normaltest

skewed_features = []
for col in numeric_features:
    valid_data = df_ema_passive[col].dropna()

    # skewtest requires sample size > 7 for reliable results
    stat, p_val = skewtest(valid_data)
    if p_val < 0.05:
        skewed_features.append(col)  # append this feature as skewed


### Model Pipeline

In [ ]:
df_ema_passive["intercept"] = 1

In [ ]:
df_ema_pipeline = df_ema_passive[['customer', 'unique_day_id', 
       'quest_create_hour', 'weekday', 'weekend', 'season', 'time_of_day',
       'n_quest', 'mean_na', 'sensor_block_end', 'age', 
       'ema_smartphone', 'psychotropic', 'somatic_problems','employability_description_simple',
       'prior_treatment_description_simple',
       'hr_mean', 'hr_min', 'hr_max', 'hr_std', 
       'hr_zone_resting', 'hr_zone_moderate',
       'hr_zone_vigorous', 'n_steps',  'n_GPS', 'total_distance_km', 'at_home_minute',
       'time_in_transition_minutes', 'time_stationary_minutes',
       'activity_102_minutes',
       'activity_103_minutes', 'activity_104_minutes', 'activity_105_minutes',
       'activity_106_minutes', 'activity_107_minutes',
       'apparent_temperature_mean', 'sunshine_duration', 'precipitation_hours', 'intercept'
      ]]

In [ ]:
df_ema_passive['customer'] = df_ema_passive['customer'].astype('string')
df_ema_passive.to_csv("short_term_data.csv")

In [ ]:
for col in numeric_features:
    series = df_ema_pipeline[col]
    n_nan = series.isna().sum()
    col_min = series.min()
    col_max = series.max()
    col_mean = series.mean()
    col_std = series.std()
    print(f"{col}: NaNs={n_nan}, min={col_min}, max={col_max}, mean={col_mean}, std={col_std}")


In [ ]:
df_ema_pipeline["n_quest_sum"] = (df_ema_pipeline.groupby('customer')['unique_day_id'].transform('nunique'))

In [ ]:
df_ema_strat = df_ema_pipeline[["customer", "n_quest_sum"]].drop_duplicates()
df_ema_strat['n_quest_stratify'] = pd.qcut(df_ema_strat["n_quest_sum"], q=4, labels=False, duplicates="drop")


In [ ]:
df_ema_pipeline= pd.merge(df_ema_pipeline,df_ema_strat, on=["customer", "n_quest_sum"], how="left")

In [ ]:
# Assume df_ema_pipeline is your DataFrame with columns "customer" and "mean_na"
# Compute individual mean outcomes.
grouped = df_ema_pipeline.groupby("customer")["mean_na"]
individual_means = grouped.mean()
overall_mean = individual_means.mean()

In [ ]:
# Assume df_ema_pipeline is your DataFrame with columns "customer" and "mean_na"
# and negative affect scores range between 1 and 7.
df_plot = df_ema_pipeline

# ---------------------------
# LEFT PANEL: Histogram of all raw negative affect scores
# ---------------------------
all_scores = df_plot["mean_na"].values
overall_mean_all = np.mean(all_scores)
overall_std_all = np.std(all_scores)
nbins = 20

# ---------------------------
# RIGHT PANEL: Density Plot of Individual Outcome Curves
# ---------------------------
data = []
group_labels = []
for customer, group in df_plot.groupby("customer"):
    if len(group["mean_na"]) > 1:
        data.append(group["mean_na"].values)
        group_labels.append(str(customer))

# Create individual density curves using Plotly's figure factory.
ff_fig = ff.create_distplot(data, group_labels, show_hist=False, show_rug=False)
ff_fig.update_traces(opacity=0.45, line=dict(width=1))

# Compute overall density using seaborn for all outcome values.
overall_kde_plot = sns.kdeplot(df_plot["mean_na"], bw_adjust=1, fill=False)
lines = overall_kde_plot.get_lines()
x_overall_ff, y_overall_ff = lines[0].get_data()
overall_kde_plot.figure.clf()  # Clear the temporary seaborn figure.

# Add the overall density as a dashed black line.
ff_fig.add_trace({
    "type": "scatter",
    "x": x_overall_ff,
    "y": y_overall_ff,
    "mode": "lines",
    "line": {"color": "black", "width": 3, "dash": "dash"},
    "name": "Overall Density"
})

# ---------------------------
# Create Subplots: 1 row, 2 columns without subplot titles.
# Set column_widths to make the right panel a bit wider.
# ---------------------------
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.45, 0.55],
    horizontal_spacing=0.05,
    subplot_titles=["", ""]  # No subplot titles.
)

# LEFT PANEL: Add histogram trace with a slightly darker blue.
fig.add_trace(go.Histogram(
    x=all_scores,
    nbinsx=nbins,
    marker_color="royalblue",  # Darker blue color.
    opacity=0.7,
    marker_line_color="black",  # Thin black borders.
    marker_line_width=1
), row=1, col=1)

# Add vertical dashed line at overall mean in the left panel.
fig.add_vline(
    x=overall_mean_all,
    line=dict(color='black', dash='dash'),
    annotation_text=f"Overall Mean = {overall_mean_all:.2f}",
    annotation_position="top right",
    annotation_xshift=20,  # Shifts the annotation further to the right.
    row=1, col=1
)

# RIGHT PANEL: Add all traces from the figure factory (density curves).
for trace in ff_fig.data:
    fig.add_trace(trace, row=1, col=2)

# ---------------------------
# Update Axes for Both Panels
# ---------------------------
fig.update_xaxes(range=[1, 7], showgrid=False, title_text="Negative Affect", row=1, col=1)
fig.update_xaxes(range=[1, 7], showgrid=False, title_text="Negative Affect", row=1, col=2)
fig.update_yaxes(showgrid=False, title_text="Count", row=1, col=1)

# For the right (density) plot, explicitly set tick values so that 0 is omitted.
fig.update_yaxes(
    showgrid=False,
    title_text="Density",
    range=[0, 3.5],
    tickmode="array",
    tickvals=[i for i in np.arange(0.5, 3.6, 0.5)],  # Ticks: 0.5, 1.0, 1.5, ..., 3.5
    row=1, col=2
)

# ---------------------------
# Update Overall Layout: Increase left margin so that the left panel's y-axis tick labels appear further left.
# ---------------------------
fig.update_layout(
    title="",
    width=1000,
    height=600,
    showlegend=False,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=120, r=20, t=20, b=20),  # Increased left margin for better spacing.
    template="plotly_white"
)


fig.show()

In [ ]:
from ML_pipeline import MLpipeline
from ML_config import Config

my_config = Config()
pipeline = MLpipeline(my_config)

pipeline.set_data(df_ema_pipeline)
pipeline.outer_user_split()
pipeline.inner_time_split()

# (1) Time-based runs
results_timebased, results_holdout, adaptation_results = pipeline.run(my_config.ANALYSIS["neg_affect_regression"]["MODEL_PIPEGRIDS"])


In [ ]:
from tensorflow.keras.utils import model_to_dot

# Assuming 'my_ffnn_regressor.model_' is your trained Keras model:
#dot = model_to_dot(my_ffnn_regressor.model_, show_shapes=True, show_layer_names=True)
#dot.write("model_architecture.dot")


In [ ]:
import json
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def format_results_as_dataframe(results_timebased, holdout_results, adaptation_results=None):
    """
    Formats the evaluation results into a structured DataFrame with columns:
      - pipeline_name
      - best_cv_score
      - inner test scores: inner_r2, inner_mae, inner_rmse
      - holdout baseline scores: holdout_r2, holdout_mae, holdout_rmse
      - adaptation scores (averaged over holdout users): adaptation_r2, adaptation_mae, adaptation_rmse
      - hyperparameters: as a formatted JSON string.
    
    Parameters
    ----------
    results_timebased : list of dict
        Each dict contains: "pipeline_name", "best_cv_score", "inner_test_scores" (with keys r2, mae, rmse),
        and "best_estimator".
    holdout_results : list of dict
        Each dict has "pipeline_name" and "holdout_scores": {r2, mae, rmse}.
    adaptation_results : dict, optional
        A mapping: adaptation_results[pipeline_name] = { user_id: {r2, mae, rmse, ...}, ... }.
        The function computes the mean r2/mae/rmse across users for that pipeline.
    
    Returns
    -------
    df_results : pd.DataFrame
        DataFrame with columns:
          [pipeline_name, best_cv_score, inner_r2, inner_mae, inner_rmse,
           holdout_r2, holdout_mae, holdout_rmse,
           adaptation_r2, adaptation_mae, adaptation_rmse,
           hyperparameters]
    """
    if adaptation_results is None:
        adaptation_results = {}
    
    # Convert holdout_results into mapping: pipeline_name -> holdout_scores.
    holdout_scores_map = {}
    if isinstance(holdout_results, list):
        for res in holdout_results:
            if isinstance(res, dict) and "pipeline_name" in res:
                holdout_scores_map[res["pipeline_name"]] = res["holdout_scores"]
    elif isinstance(holdout_results, dict):
        holdout_scores_map = holdout_results

    formatted_results = []
    
    for result in results_timebased:
        pipeline_name = result.get("pipeline_name", "Unknown")
        best_cv_score = result.get("best_cv_score", None)
        
        # Inner test scores:
        inner_scores = result.get("inner_test_scores", {})
        inner_r2   = inner_scores.get("r2", None)
        inner_mae  = inner_scores.get("mae", None)
        inner_rmse = inner_scores.get("rmse", None)
        
        # Holdout baseline scores:
        holdout_scores = holdout_scores_map.get(pipeline_name, {})
        holdout_r2   = holdout_scores.get("r2", None)
        holdout_mae  = holdout_scores.get("mae", None)
        holdout_rmse = holdout_scores.get("rmse", None)
        
        # Adaptation scores: average across users if available.
        adaptation_r2 = None
        adaptation_mae = None
        adaptation_rmse = None
        if pipeline_name in adaptation_results:
            user_dict = adaptation_results[pipeline_name]  # { user_id: {r2, mae, rmse, ...}, ... }
            if user_dict:
                df_adapt = pd.DataFrame(user_dict).T  # each row is one user's metrics
                if "r2" in df_adapt.columns:
                    adaptation_r2 = df_adapt["r2"].mean()
                if "mae" in df_adapt.columns:
                    adaptation_mae = df_adapt["mae"].mean()
                if "rmse" in df_adapt.columns:
                    adaptation_rmse = df_adapt["rmse"].mean()
        
        # Extract hyperparameters from best_estimator; store as a pretty JSON string.
        best_estimator = result.get("best_estimator", None)
        hyperparameters = ""
        if best_estimator is not None and hasattr(best_estimator, "get_params"):
            params = best_estimator.get_params(deep=True)
            try:
                hyperparameters = json.dumps(params, indent=2, default=str)
            except Exception:
                hyperparameters = str(params)
        
        formatted_results.append({
            "pipeline_name": pipeline_name,
            "best_cv_score": best_cv_score,
            "inner_r2": inner_r2,
            "inner_mae": inner_mae,
            "inner_rmse": inner_rmse,
            "holdout_r2": holdout_r2,
            "holdout_mae": holdout_mae,
            "holdout_rmse": holdout_rmse,
            "adaptation_r2": adaptation_r2,
            "adaptation_mae": adaptation_mae,
            "adaptation_rmse": adaptation_rmse,
            "hyperparameters": hyperparameters
        })
    
    df_results = pd.DataFrame(formatted_results)
    
    # Round numerical columns to 3 decimals for nicer formatting.
    num_cols = ["best_cv_score", "inner_r2", "inner_mae", "inner_rmse",
                "holdout_r2", "holdout_mae", "holdout_rmse",
                "adaptation_r2", "adaptation_mae", "adaptation_rmse"]
    for col in num_cols:
        if col in df_results.columns:
            df_results[col] = df_results[col].apply(lambda x: round(x, 3) if pd.notnull(x) else x)
    
    # Order columns as desired:
    df_results = df_results[["pipeline_name", "best_cv_score", "inner_r2", "inner_mae", "inner_rmse",
                             "holdout_r2", "holdout_mae", "holdout_rmse",
                             "adaptation_r2", "adaptation_mae", "adaptation_rmse",
                             "hyperparameters"]]
    return df_results


In [ ]:
df_results= format_results_as_dataframe(results_timebased, results_holdout, adaptation_results)

In [ ]:
df_results